# Experiment: CAFA5 Server Safe Preprocessing

Objective:
- audit the server-side preprocessing artifacts you already have
- build only the missing artifacts needed for training
- never overwrite or delete existing heavy outputs by default
- add the modified preprocessing outputs we still need:
  - a canonical aligned sequence table
  - official split and vocab manifests for sequence training
  - a raw full-sequence k-mer feature matrix
  - a 640-d protein-level ESM2 baseline derived from existing graph ESM2 caches


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

import json
import os
import random
import shlex
import subprocess
import sys
from collections import Counter
from itertools import product
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from Bio import SeqIO

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
SEED


## Safety And Plan

This notebook is intentionally conservative.

It follows these rules:

- never pass `--overwrite`
- never delete existing large outputs
- if a stage is already complete, skip it
- if a stage is partially complete, do not auto-resume it by default, because several repo scripts rewrite manifest or summary files while resuming
- only create new lightweight artifacts cell-by-cell when that can be done without touching existing files

The notebook is organized in this order:

1. audit raw inputs and current artifacts
2. build a canonical aligned sequence table if missing
3. build official sequence splits and vocabs if missing
4. build raw k-mer features if missing
5. build missing graph stages only when an entire stage is absent
6. derive matched sequence artifacts and a 640-d protein-level ESM2 baseline from existing graph caches
7. run a final audit so you know what is ready for training


In [ ]:
# Configuration
def find_repo_root(start: Path) -> tuple[Path, str]:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate, 'cwd_search'
    return start, 'cwd_fallback'

def resolve_repo_root() -> tuple[Path, str]:
    configured = os.environ.get('CAFA5_REPO_ROOT')
    if configured:
        return Path(configured).expanduser().resolve(), 'env:CAFA5_REPO_ROOT'
    return find_repo_root(Path.cwd())

REPO_ROOT, REPO_ROOT_SOURCE = resolve_repo_root()
SERVER_USER = os.environ.get('USER', 'bensonli')

TRAIN_DIR = Path(os.environ.get('CAFA5_TRAIN_DIR', f'/global/scratch/users/{SERVER_USER}/cafa5/Train'))
RUN_ROOT = Path(os.environ.get('CAFA5_RUN_ROOT', f'/global/scratch/users/{SERVER_USER}/cafa5_outputs'))

MAIN_PYTHON = Path(os.environ.get('CAFA5_MAIN_PYTHON', sys.executable))
GRAPH_PYTHON = Path(os.environ.get('CAFA5_GRAPH_PYTHON', sys.executable))
MULTIMODAL_PYTHON = Path(os.environ.get('CAFA5_MULTIMODAL_PYTHON', sys.executable))

STRICT_NO_OVERWRITE = True
RUN_MISSING_SERVER_STAGES = True
ALLOW_PARTIAL_STAGE_RESUME = False

MIN_TERM_FREQUENCY = 20
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
KMER_K = 2
GRAPH_FRAMEWORKS = ['pyg']

ALPHAFOLD_WORKERS = 4
ALPHAFOLD_REQUEST_DELAY = 0.5
FEATURE_WORKERS = 1
FEATURE_BATCH_SIZE = 5000
GRAPH_WORKERS = 1
GRAPH_BATCH_SIZE = 500

BUILD_KMER_FULL = True
BUILD_MATCHED_PROTEIN_ESM640_FROM_GRAPH_CACHE = True
BUILD_LEGACY_320D_BASELINE = False
RUN_REPORT_EXPORTS = False

TRAIN_FASTA_PATH = TRAIN_DIR / 'train_sequences.fasta'
TRAIN_TERMS_PATH = TRAIN_DIR / 'train_terms.tsv'
TRAIN_TAXONOMY_PATH = TRAIN_DIR / 'train_taxonomy.tsv'

MANIFESTS_DIR = RUN_ROOT / 'manifests'
FEATURES_DIR = RUN_ROOT / 'features'
GRAPH_CACHE_DIR = RUN_ROOT / 'graph_cache'
GRAPH_SPLIT_DIR = GRAPH_CACHE_DIR / 'splits'
GRAPH_ESM_CACHE_DIR = GRAPH_CACHE_DIR / 'modality_cache' / 'esm2'
GRAPH_STRUCTURE_CACHE_DIR = GRAPH_CACHE_DIR / 'modality_cache' / 'structure'

CANONICAL_DIR = RUN_ROOT / 'canonical_inputs'
SEQUENCE_ARTIFACT_DIR = RUN_ROOT / 'sequence_artifacts'
FULL_SPLIT_DIR = SEQUENCE_ARTIFACT_DIR / 'full_sequence_splits'
KMER_DIR = SEQUENCE_ARTIFACT_DIR / f'kmer_k{KMER_K}_full'
MATCHED_SPLIT_DIR = SEQUENCE_ARTIFACT_DIR / 'matched_structure_splits'
PROTEIN_ESM640_DIR = SEQUENCE_ARTIFACT_DIR / 'protein_esm2_t30_150m_640_from_graph_cache'

ALIGNED_PROTEINS_PATH = CANONICAL_DIR / 'aligned_proteins.parquet'
TERM_COUNTS_PATH = CANONICAL_DIR / 'term_counts.json'
CANONICAL_SUMMARY_PATH = CANONICAL_DIR / 'summary.json'

REQUIRED_REPO_FILES = [
    REPO_ROOT / 'build_esm2_cache.py',
    REPO_ROOT / 'build_structure_cache.py',
    REPO_ROOT / 'cafa_multimodal_cache_builders.py',
]
missing_repo_files = [str(path) for path in REQUIRED_REPO_FILES if not path.exists()]
if missing_repo_files:
    raise FileNotFoundError(
        'REPO_ROOT does not point to the CAFA5 repo. '
        f'repo_root={REPO_ROOT} (source={REPO_ROOT_SOURCE}). Missing files: {missing_repo_files}. '
        'Set CAFA5_REPO_ROOT to your checked-out repo path, then rerun this cell.'
    )

config = {
    'repo_root': str(REPO_ROOT),
    'repo_root_source': REPO_ROOT_SOURCE,
    'train_dir': str(TRAIN_DIR),
    'run_root': str(RUN_ROOT),
    'main_python': str(MAIN_PYTHON),
    'graph_python': str(GRAPH_PYTHON),
    'multimodal_python': str(MULTIMODAL_PYTHON),
    'strict_no_overwrite': STRICT_NO_OVERWRITE,
    'run_missing_server_stages': RUN_MISSING_SERVER_STAGES,
    'allow_partial_stage_resume': ALLOW_PARTIAL_STAGE_RESUME,
    'min_term_frequency': MIN_TERM_FREQUENCY,
    'kmer_k': KMER_K,
}
print(json.dumps(config, indent=2))


In [ ]:
# Lightweight helpers
ASPECTS = ('BPO', 'CCO', 'MFO')
VALID_AAS = set('ACDEFGHIKLMNPQRSTVWY')

def progress(message: str) -> None:
    print(f'[progress] {message}', flush=True)

def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def load_json(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def write_json_if_missing(path: Path, payload) -> None:
    if path.exists():
        print(f'[skip] JSON already exists: {path}')
        return
    ensure_parent(path)
    with path.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2)
    print(f'[write] {path}')

def write_lines_if_missing(path: Path, lines: list[str]) -> None:
    if path.exists():
        print(f'[skip] text file already exists: {path}')
        return
    ensure_parent(path)
    with path.open('w', encoding='utf-8') as handle:
        for line in lines:
            handle.write(f'{line}\n')
    print(f'[write] {path}')

def save_npy_if_missing(path: Path, array: np.ndarray) -> None:
    if path.exists():
        print(f'[skip] NPY already exists: {path}')
        return
    ensure_parent(path)
    np.save(path, array)
    print(f'[write] {path}')

def count_pt_files(
    directory: Path,
    label: str | None = None,
    progress_every: int = 5000,
    show_progress: bool = False,
) -> int:
    if not directory.exists():
        if show_progress and label:
            progress(f'{label}: directory missing -> 0 files')
        return 0
    count = 0
    started_at = perf_counter()
    if show_progress and label:
        progress(f'{label}: scanning {directory}')
    for item in directory.iterdir():
        if item.is_file() and item.suffix == '.pt':
            count += 1
            if show_progress and label and progress_every > 0 and count % progress_every == 0:
                elapsed = perf_counter() - started_at
                progress(f'{label}: counted {count:,} .pt files after {elapsed:.1f}s')
    if show_progress and label:
        elapsed = perf_counter() - started_at
        progress(f'{label}: finished with {count:,} .pt files in {elapsed:.1f}s')
    return count

def classify_paths(paths: list[Path]) -> str:
    exists = [path.exists() for path in paths]
    if all(exists):
        return 'complete'
    if not any(exists):
        return 'missing'
    return 'partial'

def graph_cache_state(graph_root: Path, show_progress: bool = False) -> tuple[str, dict[str, int]]:
    graphs_dir = graph_root / 'graphs'
    metadata_dir = graph_root / 'metadata'
    entries_path = metadata_dir / 'entries.json'
    schema_path = metadata_dir / 'schema.json'
    term_counts_path = metadata_dir / 'term_counts.json'
    graph_count = count_pt_files(
        graphs_dir,
        label='graph_cache/graphs',
        show_progress=show_progress,
    )
    entries_count = 0
    if entries_path.exists():
        if show_progress:
            progress(f'graph_cache/metadata: loading {entries_path}')
        entries_count = len(load_json(entries_path))
        if show_progress:
            progress(f'graph_cache/metadata: entries.json has {entries_count:,} records')
    metadata_exists = [entries_path.exists(), schema_path.exists(), term_counts_path.exists()]
    if graph_count == 0 and not any(metadata_exists):
        return 'missing', {'graphs': 0, 'entries_json': 0}
    if all(metadata_exists) and entries_count > 0 and graph_count == entries_count:
        return 'complete', {'graphs': graph_count, 'entries_json': entries_count}
    return 'partial', {'graphs': graph_count, 'entries_json': entries_count}

def modality_cache_state(
    cache_dir: Path,
    expected_count: int,
    label: str = 'modality_cache',
    show_progress: bool = False,
) -> tuple[str, dict[str, int]]:
    pt_count = count_pt_files(
        cache_dir,
        label=label,
        show_progress=show_progress,
    )
    if expected_count <= 0:
        return ('complete' if pt_count > 0 else 'missing'), {'pt_files': pt_count, 'expected': expected_count}
    if pt_count == 0:
        return 'missing', {'pt_files': 0, 'expected': expected_count}
    if pt_count >= expected_count:
        return 'complete', {'pt_files': pt_count, 'expected': expected_count}
    return 'partial', {'pt_files': pt_count, 'expected': expected_count}

def graph_split_state(split_root: Path, show_progress: bool = False) -> str:
    if show_progress:
        progress(f'graph_splits: checking required files under {split_root}')
    required = [split_root / 'summary.json', split_root / 'export_summary.json']
    for aspect in ('bpo', 'cco', 'mfo'):
        required.extend([
            split_root / aspect / 'train.txt',
            split_root / aspect / 'val.txt',
            split_root / aspect / 'test.txt',
            split_root / aspect / 'summary.json',
        ])
    return classify_paths(required)

def format_cmd(cmd: list[object]) -> str:
    return ' '.join(shlex.quote(str(part)) for part in cmd)

def run_command(cmd: list[object], cwd: Path | None = None) -> None:
    cmd_str = format_cmd(cmd)
    if STRICT_NO_OVERWRITE and '--overwrite' in {str(part) for part in cmd}:
        raise RuntimeError(f'Refusing to run command with --overwrite: {cmd_str}')
    print(f'[run] {cmd_str}')
    subprocess.run([str(part) for part in cmd], cwd=str(cwd or REPO_ROOT), check=True)

def maybe_run_server_stage(label: str, state: str, cmd: list[object] | None = None) -> None:
    print(f'[{label}] state = {state}')
    if state == 'complete':
        print(f'[{label}] already complete -> skip')
        return
    if state == 'partial' and not ALLOW_PARTIAL_STAGE_RESUME:
        print(f'[{label}] partial stage detected -> skip for safety (no overwrite / no auto-resume).')
        return
    if state == 'missing' and not RUN_MISSING_SERVER_STAGES:
        print(f'[{label}] missing stage detected -> configured to report only, not run.')
        return
    if cmd is None:
        print(f'[{label}] no command attached.')
        return
    run_command(cmd)

def allocate_split_counts(total_items: int, ratios: tuple[float, float, float]) -> tuple[int, int, int]:
    expected = [ratio * total_items for ratio in ratios]
    counts = [int(value) for value in expected]
    remainder = total_items - sum(counts)
    order = sorted(
        range(len(expected)),
        key=lambda index: (expected[index] - counts[index], ratios[index], -index),
        reverse=True,
    )
    for index in range(remainder):
        counts[order[index % len(order)]] += 1

    positive_indices = [index for index, ratio in enumerate(ratios) if ratio > 0]
    if total_items >= len(positive_indices):
        for index in positive_indices:
            if counts[index] > 0:
                continue
            donor = max(
                (candidate for candidate in range(len(counts)) if counts[candidate] > 1),
                key=lambda candidate: counts[candidate],
                default=None,
            )
            if donor is None:
                break
            counts[donor] -= 1
            counts[index] += 1
    return counts[0], counts[1], counts[2]

def split_entry_ids(entry_ids: list[str], seed: int = SEED) -> dict[str, list[str]]:
    unique_ids = sorted({str(entry_id) for entry_id in entry_ids if str(entry_id).strip()})
    rng = random.Random(seed)
    rng.shuffle(unique_ids)
    train_count, val_count, test_count = allocate_split_counts(
        len(unique_ids),
        (TRAIN_RATIO, VAL_RATIO, TEST_RATIO),
    )
    return {
        'train': unique_ids[:train_count],
        'val': unique_ids[train_count: train_count + val_count],
        'test': unique_ids[train_count + val_count: train_count + val_count + test_count],
    }


## Audit Current Inputs And Artifacts

This cell is only diagnostic. It checks:

- raw CAFA5 files
- AlphaFold manifest stage
- feature extraction stage
- graph cache stage
- modality cache stage
- graph split stage
- the new sequence artifact folders introduced by this notebook


In [ ]:
# Stage inventory
def count_parquet_matches(
    path: Path,
    column: str,
    expected_value: str,
    label: str,
    batch_size: int = 250_000,
    progress_every_batches: int = 10,
    show_progress: bool = False,
) -> int:
    if not path.exists():
        if show_progress:
            progress(f'{label}: manifest missing -> 0 matches')
        return 0
    parquet_file = pq.ParquetFile(path)
    total_matches = 0
    total_rows = 0
    started_at = perf_counter()
    if show_progress:
        progress(
            f'{label}: scanning {path.name} with {parquet_file.num_row_groups} row groups '
            f'(batch_size={batch_size:,})'
        )
    for batch_index, batch in enumerate(parquet_file.iter_batches(columns=[column], batch_size=batch_size), start=1):
        matches_scalar = pc.sum(pc.cast(pc.equal(batch.column(0), expected_value), pa.int64()))
        batch_matches = int(matches_scalar.as_py() or 0)
        total_matches += batch_matches
        total_rows += batch.num_rows
        if show_progress and (batch_index == 1 or batch_index % progress_every_batches == 0):
            progress(
                f'{label}: processed {total_rows:,} rows across {batch_index} batches; '
                f'matched {total_matches:,}'
            )
    if show_progress:
        elapsed = perf_counter() - started_at
        progress(f'{label}: finished {total_rows:,} rows; matched {total_matches:,} in {elapsed:.1f}s')
    return total_matches

def summarize_expected_counts(show_progress: bool = False) -> tuple[int, int]:
    expected_ok_entries = 0
    expected_ok_fragments = 0
    training_manifest = MANIFESTS_DIR / 'training_index.parquet'
    fragment_manifest = MANIFESTS_DIR / 'alphafold_fragments.parquet'
    expected_ok_entries = count_parquet_matches(
        training_manifest,
        column='af_status',
        expected_value='ok',
        label='training_index.af_status',
        show_progress=show_progress,
    )
    expected_ok_fragments = count_parquet_matches(
        fragment_manifest,
        column='fragment_status',
        expected_value='ok',
        label='alphafold_fragments.fragment_status',
        show_progress=show_progress,
    )
    return expected_ok_entries, expected_ok_fragments

def build_stage_inventory(show_progress: bool = False) -> pd.DataFrame:
    raw_inputs = [TRAIN_FASTA_PATH, TRAIN_TERMS_PATH, TRAIN_TAXONOMY_PATH]
    manifest_paths = [
        MANIFESTS_DIR / 'training_index.parquet',
        MANIFESTS_DIR / 'alphafold_fragments.parquet',
        MANIFESTS_DIR / 'download_failures.csv',
    ]
    feature_paths = [
        FEATURES_DIR / 'fragment_features.parquet',
        FEATURES_DIR / 'residue_features.parquet',
        FEATURES_DIR / 'contact_graph_edges.parquet',
    ]

    if show_progress:
        progress('inventory: checking raw input, manifest, and feature file presence')
    raw_input_details = {path.name: path.exists() for path in raw_inputs}
    manifest_details = {path.name: path.exists() for path in manifest_paths}
    feature_details = {path.name: path.exists() for path in feature_paths}

    if show_progress:
        progress('inventory: counting expected ok entries and ok fragments from parquet manifests')
    expected_ok_entries, expected_ok_fragments = summarize_expected_counts(show_progress=show_progress)

    if show_progress:
        progress('inventory: scanning graph cache')
    graph_state, graph_meta = graph_cache_state(GRAPH_CACHE_DIR, show_progress=show_progress)

    if show_progress:
        progress('inventory: scanning graph ESM cache')
    esm_state, esm_meta = modality_cache_state(
        GRAPH_ESM_CACHE_DIR,
        expected_ok_entries,
        label='graph_esm_cache',
        show_progress=show_progress,
    )

    if show_progress:
        progress('inventory: scanning graph structure cache')
    structure_state, structure_meta = modality_cache_state(
        GRAPH_STRUCTURE_CACHE_DIR,
        expected_ok_fragments,
        label='graph_structure_cache',
        show_progress=show_progress,
    )

    if show_progress:
        progress('inventory: checking graph split files')
    split_state = graph_split_state(GRAPH_SPLIT_DIR, show_progress=show_progress)

    stage_rows = [
        {'stage': 'raw_inputs', 'state': classify_paths(raw_inputs), 'details': raw_input_details},
        {'stage': 'alphafold_manifests', 'state': classify_paths(manifest_paths), 'details': manifest_details},
        {'stage': 'feature_tables', 'state': classify_paths(feature_paths), 'details': feature_details},
        {'stage': 'graph_cache', 'state': graph_state, 'details': graph_meta},
        {'stage': 'graph_esm_cache', 'state': esm_state, 'details': esm_meta},
        {'stage': 'graph_structure_cache', 'state': structure_state, 'details': structure_meta},
        {'stage': 'graph_splits', 'state': split_state, 'details': {'root': str(GRAPH_SPLIT_DIR)}},
        {'stage': 'canonical_inputs', 'state': classify_paths([ALIGNED_PROTEINS_PATH, TERM_COUNTS_PATH, CANONICAL_SUMMARY_PATH]), 'details': {'root': str(CANONICAL_DIR)}},
        {'stage': 'full_sequence_splits', 'state': 'complete' if (FULL_SPLIT_DIR / 'summary.json').exists() else 'missing', 'details': {'root': str(FULL_SPLIT_DIR)}},
        {'stage': 'raw_kmer_features', 'state': classify_paths([KMER_DIR / 'X.npy', KMER_DIR / 'entry_ids.txt', KMER_DIR / 'meta.json']), 'details': {'root': str(KMER_DIR)}},
        {'stage': 'matched_structure_splits', 'state': 'complete' if (MATCHED_SPLIT_DIR / 'summary.json').exists() else 'missing', 'details': {'root': str(MATCHED_SPLIT_DIR)}},
        {'stage': 'protein_esm2_640_from_graph_cache', 'state': classify_paths([PROTEIN_ESM640_DIR / 'X.npy', PROTEIN_ESM640_DIR / 'entry_ids.txt', PROTEIN_ESM640_DIR / 'meta.json']), 'details': {'root': str(PROTEIN_ESM640_DIR)}},
    ]
    if show_progress:
        progress('inventory: stage summary ready')
    return pd.DataFrame(stage_rows)

stage_inventory_df = build_stage_inventory(show_progress=True)
stage_inventory_df


## Canonical Aligned Sequence Dataset

This is the main sequence-side normalization step added by this notebook.

It creates a canonical table with one row per usable protein:

- `entry_id`
- `taxonomy_id`
- `sequence`
- `sequence_length`
- `go_terms_bpo`
- `go_terms_cco`
- `go_terms_mfo`

This cell only writes the canonical files if they do not already exist.


In [ ]:
# Build canonical aligned sequence table if missing
def load_clean_sequences(fasta_path: Path) -> tuple[dict[str, str], dict[str, int]]:
    clean_sequences = {}
    removed_invalid = 0
    removed_empty = 0
    for record in SeqIO.parse(str(fasta_path), 'fasta'):
        entry_id = record.id
        sequence = str(record.seq).strip()
        if not sequence:
            removed_empty += 1
            continue
        if set(sequence) - VALID_AAS:
            removed_invalid += 1
            continue
        clean_sequences[entry_id] = sequence
    return clean_sequences, {
        'removed_invalid_sequences': removed_invalid,
        'removed_empty_sequences': removed_empty,
        'clean_sequences': len(clean_sequences),
    }

if ALIGNED_PROTEINS_PATH.exists():
    print(f'[skip] canonical table already exists: {ALIGNED_PROTEINS_PATH}')
else:
    taxonomy = pd.read_csv(TRAIN_TAXONOMY_PATH, sep='\t')
    terms = pd.read_csv(TRAIN_TERMS_PATH, sep='\t').dropna(subset=['EntryID', 'term', 'aspect']).copy()
    sequences, sequence_stats = load_clean_sequences(TRAIN_FASTA_PATH)

    aspect_maps = {}
    term_counts = {}
    for aspect in ASPECTS:
        grouped = (
            terms.loc[terms['aspect'] == aspect, ['EntryID', 'term']]
            .groupby('EntryID')['term']
            .apply(lambda values: sorted(set(map(str, values))))
            .to_dict()
        )
        aspect_maps[aspect] = grouped
        counter = Counter(term for labels in grouped.values() for term in labels)
        term_counts[aspect] = dict(counter)

    rows = []
    for row in taxonomy.itertuples(index=False):
        entry_id = str(row.EntryID)
        sequence = sequences.get(entry_id)
        if sequence is None:
            continue
        bpo = aspect_maps['BPO'].get(entry_id, [])
        cco = aspect_maps['CCO'].get(entry_id, [])
        mfo = aspect_maps['MFO'].get(entry_id, [])
        if not (bpo or cco or mfo):
            continue
        rows.append({
            'entry_id': entry_id,
            'taxonomy_id': str(row.taxonomyID),
            'sequence': sequence,
            'sequence_length': len(sequence),
            'go_terms_bpo': bpo,
            'go_terms_cco': cco,
            'go_terms_mfo': mfo,
        })

    ensure_parent(ALIGNED_PROTEINS_PATH)
    pq.write_table(pa.Table.from_pylist(rows), ALIGNED_PROTEINS_PATH)
    write_json_if_missing(TERM_COUNTS_PATH, term_counts)
    write_json_if_missing(
        CANONICAL_SUMMARY_PATH,
        {
            'row_count': len(rows),
            'sequence_stats': sequence_stats,
            'proteins_with_labels': {
                aspect: int(sum(1 for item in rows if item[f'go_terms_{aspect.lower()}']))
                for aspect in ASPECTS
            },
        },
    )

canonical_df = pq.read_table(ALIGNED_PROTEINS_PATH).to_pandas()
canonical_summary = load_json(CANONICAL_SUMMARY_PATH) if CANONICAL_SUMMARY_PATH.exists() else {}
print('canonical rows:', len(canonical_df))
print(json.dumps(canonical_summary, indent=2)[:1500])
canonical_df.head(3)


## Official Full-Sequence Splits And Vocab Files

This is a modified preprocessing step compared with the older notebooks.

Instead of saving one huge dense label matrix and a single random 80/20 split, this cell freezes:

- `train.txt`
- `val.txt`
- `test.txt`
- `vocab.json`
- `summary.json`

for each GO aspect separately.


In [ ]:
# Build full-sequence split and vocab artifacts if missing
term_counts = load_json(TERM_COUNTS_PATH)
split_top_level = {
    'root': str(FULL_SPLIT_DIR),
    'seed': SEED,
    'min_term_frequency': MIN_TERM_FREQUENCY,
    'aspects': {},
}

for aspect in ASPECTS:
    aspect_dir = FULL_SPLIT_DIR / aspect.lower()
    summary_path = aspect_dir / 'summary.json'
    vocab_path = aspect_dir / 'vocab.json'
    train_path = aspect_dir / 'train.txt'
    val_path = aspect_dir / 'val.txt'
    test_path = aspect_dir / 'test.txt'

    label_column = f'go_terms_{aspect.lower()}'
    vocab = sorted(
        term for term, count in (term_counts.get(aspect) or {}).items()
        if int(count) >= MIN_TERM_FREQUENCY
    )
    vocab_set = set(vocab)
    eligible_ids = [
        str(row.entry_id)
        for row in canonical_df.itertuples(index=False)
        if vocab_set.intersection(getattr(row, label_column))
    ]
    splits = split_entry_ids(eligible_ids, seed=SEED)

    write_json_if_missing(vocab_path, {
        'aspect': aspect,
        'min_term_frequency': MIN_TERM_FREQUENCY,
        'terms': vocab,
    })
    write_lines_if_missing(train_path, splits['train'])
    write_lines_if_missing(val_path, splits['val'])
    write_lines_if_missing(test_path, splits['test'])
    write_json_if_missing(summary_path, {
        'aspect': aspect,
        'entry_count': len(eligible_ids),
        'vocab_size': len(vocab),
        'seed': SEED,
        'train_ratio': TRAIN_RATIO,
        'val_ratio': VAL_RATIO,
        'test_ratio': TEST_RATIO,
        'counts': {name: len(ids) for name, ids in splits.items()},
    })

    split_top_level['aspects'][aspect] = {
        'entry_count': len(eligible_ids),
        'vocab_size': len(vocab),
        'counts': {name: len(ids) for name, ids in splits.items()},
    }

write_json_if_missing(FULL_SPLIT_DIR / 'summary.json', split_top_level)
split_top_level


## Raw Full-Sequence K-mer Features

This notebook changes the older k-mer preprocessing slightly.

Instead of saving train and val feature matrices only, it saves a full raw feature matrix plus IDs and metadata. This is safer because:

- split definitions are frozen separately
- scaling can be fit on the training IDs only during model training
- we avoid duplicating labels into another huge dense output


In [ ]:
# Build raw normalized k-mer feature matrix if missing
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
kmer_vocab = [''.join(chunk) for chunk in product(amino_acids, repeat=KMER_K)]
kmer_to_index = {kmer: index for index, kmer in enumerate(kmer_vocab)}

def seq_to_kmer_vector(sequence: str) -> np.ndarray:
    vector = np.zeros(len(kmer_vocab), dtype=np.float32)
    total = len(sequence) - KMER_K + 1
    if total <= 0:
        return vector
    for index in range(total):
        token = sequence[index:index + KMER_K]
        matched = kmer_to_index.get(token)
        if matched is not None:
            vector[matched] += 1.0
    vector /= total
    return vector

if BUILD_KMER_FULL:
    x_path = KMER_DIR / 'X.npy'
    ids_path = KMER_DIR / 'entry_ids.txt'
    meta_path = KMER_DIR / 'meta.json'

    if not x_path.exists():
        print('Computing raw full-sequence k-mer features...')
        matrix = np.stack([seq_to_kmer_vector(seq) for seq in canonical_df['sequence']], axis=0)
        save_npy_if_missing(x_path, matrix.astype(np.float32))
    else:
        print(f'[skip] k-mer matrix already exists: {x_path}')

    write_lines_if_missing(ids_path, canonical_df['entry_id'].astype(str).tolist())
    write_json_if_missing(meta_path, {
        'k': KMER_K,
        'feature_count': len(kmer_vocab),
        'entry_count': int(len(canonical_df)),
        'vocab': kmer_vocab,
        'source': str(ALIGNED_PROTEINS_PATH),
    })

    kmer_x = np.load(x_path, mmap_mode='r')
    print('k-mer shape:', kmer_x.shape, 'dtype:', kmer_x.dtype)
else:
    print('BUILD_KMER_FULL = False -> skipped')


## Graph Stage 1: AlphaFold Manifests

This cell is safe by default:

- if manifests are complete, it skips
- if manifests are partially present, it also skips by default
- it only auto-runs when the entire manifest stage is absent


In [ ]:
manifest_state = classify_paths([
    MANIFESTS_DIR / 'training_index.parquet',
    MANIFESTS_DIR / 'alphafold_fragments.parquet',
    MANIFESTS_DIR / 'download_failures.csv',
])

manifest_cmd = [
    MAIN_PYTHON,
    REPO_ROOT / 'cafa5_alphafold_pipeline.py',
    '--train-taxonomy', TRAIN_TAXONOMY_PATH,
    '--train-sequences', TRAIN_FASTA_PATH,
    '--train-terms', TRAIN_TERMS_PATH,
    '--output-dir', RUN_ROOT,
    '--workers', ALPHAFOLD_WORKERS,
    '--request-delay', ALPHAFOLD_REQUEST_DELAY,
    '--resume',
]

maybe_run_server_stage('AlphaFold manifests', manifest_state, manifest_cmd)


## Graph Stage 2: Feature Tables

This stage writes:

- `fragment_features.parquet`
- `residue_features.parquet`
- `contact_graph_edges.parquet`

The same strict safety rule applies: partial stages are not auto-resumed unless you explicitly change the config.


In [ ]:
feature_state = classify_paths([
    FEATURES_DIR / 'fragment_features.parquet',
    FEATURES_DIR / 'residue_features.parquet',
    FEATURES_DIR / 'contact_graph_edges.parquet',
])

feature_cmd = [
    MAIN_PYTHON,
    REPO_ROOT / 'alphafold_feature_extractor.py',
    '--training-index', MANIFESTS_DIR / 'training_index.parquet',
    '--fragment-manifest', MANIFESTS_DIR / 'alphafold_fragments.parquet',
    '--output-dir', FEATURES_DIR,
    '--workers', FEATURE_WORKERS,
    '--batch-size', FEATURE_BATCH_SIZE,
    '--resume',
]

maybe_run_server_stage('Feature extraction', feature_state, feature_cmd)


## Graph Stage 3: Graph Cache

This stage is treated as complete only when:

- `entries.json`, `schema.json`, and `term_counts.json` exist
- the number of graph `.pt` files matches the number of entries in `entries.json`


In [ ]:
graph_state, graph_meta = graph_cache_state(GRAPH_CACHE_DIR)

graph_cmd = [
    GRAPH_PYTHON,
    REPO_ROOT / 'build_cafa_graph_cache.py',
    '--training-index', MANIFESTS_DIR / 'training_index.parquet',
    '--fragment-features', FEATURES_DIR / 'fragment_features.parquet',
    '--residue-features', FEATURES_DIR / 'residue_features.parquet',
    '--edge-features', FEATURES_DIR / 'contact_graph_edges.parquet',
    '--output-dir', GRAPH_CACHE_DIR,
    '--min-term-frequency', MIN_TERM_FREQUENCY,
    '--workers', GRAPH_WORKERS,
    '--batch-size', GRAPH_BATCH_SIZE,
    '--resume',
]

print(graph_meta)
maybe_run_server_stage('Graph cache', graph_state, graph_cmd)


## Graph Stage 4: Modality Caches

This notebook only auto-runs a modality stage if the cache directory is entirely missing.

If some cache files already exist but the cache is incomplete, the notebook stops short of auto-resuming, because the builder scripts still rewrite summary files.


In [ ]:
expected_ok_entries, expected_ok_fragments = summarize_expected_counts()
esm_state, esm_meta = modality_cache_state(GRAPH_ESM_CACHE_DIR, expected_ok_entries)
structure_state, structure_meta = modality_cache_state(GRAPH_STRUCTURE_CACHE_DIR, expected_ok_fragments)

esm_cmd = [
    MULTIMODAL_PYTHON,
    REPO_ROOT / 'build_esm2_cache.py',
    '--training-index', MANIFESTS_DIR / 'training_index.parquet',
    '--output-dir', GRAPH_ESM_CACHE_DIR,
    '--resume',
]
structure_cmd = [
    MULTIMODAL_PYTHON,
    REPO_ROOT / 'build_structure_cache.py',
    '--fragment-manifest', MANIFESTS_DIR / 'alphafold_fragments.parquet',
    '--output-dir', GRAPH_STRUCTURE_CACHE_DIR,
    '--resume',
]

print('ESM cache:', esm_meta)
maybe_run_server_stage('Graph residue ESM2 cache', esm_state, esm_cmd)
print('Structure cache:', structure_meta)
maybe_run_server_stage('Graph structure cache', structure_state, structure_cmd)


## Graph Stage 5: Graph Splits

The export script writes split manifests and summary files. To preserve the no-overwrite rule, this cell only auto-runs when the split stage is entirely absent.


In [ ]:
split_state = graph_split_state(GRAPH_SPLIT_DIR)
split_cmd = [
    GRAPH_PYTHON,
    REPO_ROOT / 'export_graph_dataloaders.py',
    '--root', GRAPH_CACHE_DIR,
    '--aspects', 'BPO', 'CCO', 'MFO',
    '--min-term-frequency', MIN_TERM_FREQUENCY,
    '--frameworks', *GRAPH_FRAMEWORKS,
    '--batch-size', 8,
]
maybe_run_server_stage('Graph splits', split_state, split_cmd)


## Matched Structure-Cohort Sequence Artifacts

These are the main new sequence-side artifacts for fair comparison with graph models.

This section does two things:

- mirrors the graph split IDs into a sequence-artifact folder for matched experiments
- builds a `640-d` protein-level ESM2 baseline by reusing `protein_embedding` from the existing graph ESM2 caches


In [ ]:
# Mirror graph split IDs into a sequence-side folder, without overwriting
if (GRAPH_SPLIT_DIR / 'summary.json').exists():
    matched_summary = {
        'source_graph_splits': str(GRAPH_SPLIT_DIR),
        'source_graph_vocab_dir': str(GRAPH_CACHE_DIR / 'metadata' / 'vocabs'),
        'root': str(MATCHED_SPLIT_DIR),
        'min_term_frequency': MIN_TERM_FREQUENCY,
        'aspects': {},
    }
    for aspect in ('bpo', 'cco', 'mfo'):
        src_dir = GRAPH_SPLIT_DIR / aspect
        dst_dir = MATCHED_SPLIT_DIR / aspect
        if not src_dir.exists():
            continue
        for name in ('train.txt', 'val.txt', 'test.txt', 'summary.json'):
            src = src_dir / name
            dst = dst_dir / name
            if src.exists() and not dst.exists():
                ensure_parent(dst)
                dst.write_text(src.read_text(encoding='utf-8'), encoding='utf-8')
                print(f'[write] {dst}')
            elif dst.exists():
                print(f'[skip] already exists: {dst}')
        vocab_src = GRAPH_CACHE_DIR / 'metadata' / 'vocabs' / f'{aspect.upper()}.json'
        vocab_dst = dst_dir / 'vocab.json'
        if vocab_src.exists() and not vocab_dst.exists():
            ensure_parent(vocab_dst)
            vocab_dst.write_text(vocab_src.read_text(encoding='utf-8'), encoding='utf-8')
            print(f'[write] {vocab_dst}')
        elif vocab_dst.exists():
            print(f'[skip] already exists: {vocab_dst}')
        matched_summary['aspects'][aspect.upper()] = {'source': str(src_dir)}
    write_json_if_missing(MATCHED_SPLIT_DIR / 'summary.json', matched_summary)
else:
    print('Graph split summary is missing -> matched structure splits were not built.')

# Build 640-d protein-level ESM2 baseline from existing graph ESM2 caches
if BUILD_MATCHED_PROTEIN_ESM640_FROM_GRAPH_CACHE:
    expected_ok_entries, _ = summarize_expected_counts()
    esm_state, esm_meta = modality_cache_state(GRAPH_ESM_CACHE_DIR, expected_ok_entries)
    if esm_state != 'complete':
        print(f'Graph ESM cache is {esm_state}; skip protein-level 640-d export for safety.')
    else:
        x_path = PROTEIN_ESM640_DIR / 'X.npy'
        ids_path = PROTEIN_ESM640_DIR / 'entry_ids.txt'
        meta_path = PROTEIN_ESM640_DIR / 'meta.json'

        if not x_path.exists():
            import torch

            cache_paths = sorted(
                path for path in GRAPH_ESM_CACHE_DIR.iterdir()
                if path.is_file() and path.suffix == '.pt'
            )
            entry_ids = []
            embeddings = []
            for cache_path in cache_paths:
                payload = torch.load(cache_path, map_location='cpu', weights_only=False)
                protein_embedding = payload.get('protein_embedding')
                entry_id = str(payload.get('entry_id') or cache_path.stem)
                if protein_embedding is None:
                    continue
                entry_ids.append(entry_id)
                embeddings.append(np.asarray(protein_embedding, dtype=np.float32))

            matrix = np.stack(embeddings, axis=0)
            save_npy_if_missing(x_path, matrix)
            write_lines_if_missing(ids_path, entry_ids)
            write_json_if_missing(meta_path, {
                'entry_count': len(entry_ids),
                'embedding_dim': int(matrix.shape[1]),
                'source_cache_dir': str(GRAPH_ESM_CACHE_DIR),
                'model_name': 'facebook/esm2_t30_150M_UR50D',
                'notes': 'Derived from existing graph ESM2 caches. No recomputation from raw sequence.',
            })
        else:
            print(f'[skip] protein-level 640-d matrix already exists: {x_path}')

        protein_x = np.load(x_path, mmap_mode='r')
        print('protein-level 640-d shape:', protein_x.shape, 'dtype:', protein_x.dtype)
else:
    print('BUILD_MATCHED_PROTEIN_ESM640_FROM_GRAPH_CACHE = False -> skipped')


## Optional Report Exports

These outputs are not required for training. They are turned off by default to avoid rewriting existing summaries.


In [ ]:
if RUN_REPORT_EXPORTS:
    report_json = RUN_ROOT / 'exploration_report.json'
    report_md = RUN_ROOT / 'exploration_report.md'
    report_state = classify_paths([report_json, report_md])
    report_cmd = [
        MAIN_PYTHON,
        REPO_ROOT / 'explore_cafa_data.py',
        '--root', RUN_ROOT,
        '--output-json', report_json,
        '--output-md', report_md,
    ]
    maybe_run_server_stage('Exploration report', report_state, report_cmd)
else:
    print('RUN_REPORT_EXPORTS = False -> skipped')


## Final Audit

The final cell recomputes the stage inventory so you can see exactly what is training-ready.


In [ ]:
final_inventory_df = build_stage_inventory(show_progress=True)
final_inventory_df


## Next Step After This Notebook

If the final audit shows these are complete:

- canonical inputs
- full-sequence splits
- raw k-mer features
- graph cache
- graph splits
- matched structure splits
- protein-level 640-d ESM2 from graph cache

then you are ready to move into model training without re-running the old notebooks.

Recommended training order:

1. k-mer baseline on full-sequence cohort
2. 640-d protein-level ESM2 baseline on matched structure cohort
3. graph baseline
4. graph + residue ESM2
5. graph full multimodal
